# Emotion Recognition: TEO vs MFCC

Classify speech emotions using two feature representations from pyvoicebox:

- **MFCC** (`v_melcepst`) — spectral envelope features
- **TEO** (`v_teager`) — Teager energy features, sensitive to nonlinear vocal fold dynamics

Dataset: [EmoDB](http://emodb.bilderbar.info/) — Berlin Database of Emotional Speech. We classify three emotions: **anger**, **happiness**, and **sadness**.

In [ ]:
# Install dependencies if needed
# %pip install pyvoicebox scikit-learn pandas

In [ ]:
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display

from pyvoicebox import v_melcepst, v_teager, v_enframe, v_windows
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Download and prepare EmoDB

In [ ]:
data_dir = "emodb"
zip_path = "emodb.zip"
wav_dir = os.path.join(data_dir, "wav")

if not os.path.isdir(wav_dir):
    if not os.path.exists(zip_path):
        print("Downloading EmoDB (~26 MB)...")
        urllib.request.urlretrieve("http://emodb.bilderbar.info/download/download.zip", zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_dir)
    print("Done.")
else:
    print(f"EmoDB already extracted in {wav_dir}")

In [ ]:
# Parse filenames: emotion is the 6th character
# W=anger, L=boredom, E=disgust, A=fear, F=happiness, T=sadness, N=neutral
emotion_map = {
    'W': 'anger', 'L': 'boredom', 'E': 'disgust',
    'A': 'fear', 'F': 'happiness', 'T': 'sadness', 'N': 'neutral'
}

rows = []
for fname in sorted(os.listdir(wav_dir)):
    if not fname.endswith('.wav'):
        continue
    fpath = os.path.join(wav_dir, fname)
    info = sf.info(fpath)
    emo_code = fname[5]  # 6th character (0-indexed)
    rows.append({
        'filename': fname,
        'filepath': fpath,
        'emotion': emotion_map.get(emo_code, '?'),
        'speaker_id': fname[:2],
        'duration_seconds': info.duration,
        'sample_rate': info.samplerate,
    })

df = pd.DataFrame(rows)

# Filter to anger, happiness, sadness
target_emotions = ['anger', 'happiness', 'sadness']
df = df[df['emotion'].isin(target_emotions)].reset_index(drop=True)
df.to_csv('emodb_index.csv', index=False)

print(f"Total files: {len(df)}")
print(f"Sample rate:  {df['sample_rate'].unique()}")
print(f"Duration:     {df['duration_seconds'].min():.2f}s / {df['duration_seconds'].mean():.2f}s / {df['duration_seconds'].max():.2f}s  (min/mean/max)")
print()
print(df['emotion'].value_counts().sort_index())

## Listen to examples

One sample per emotion:

In [ ]:
for emo in sorted(df['emotion'].unique()):
    row = df[df['emotion'] == emo].iloc[0]
    sig, rate = sf.read(row['filepath'])
    print(f"{emo} ({row['filename']}, {row['duration_seconds']:.1f}s):")
    display(Audio(sig, rate=rate))

## Feature extraction

We extract three feature sets per utterance:

1. **MFCC** — 13 coefficients via `v_melcepst`. Captures the spectral envelope.
2. **TEO** — Teager energy via `v_teager` applied per frame, then summarised in sub-bands using a filterbank. Captures nonlinear excitation energy.
3. **MFCC + TEO** — concatenation of both.

For each, we compute per-utterance statistics (mean and standard deviation across frames) to get a fixed-length feature vector regardless of utterance duration.

In [ ]:
def extract_mfcc_features(signal, fs, n_cep=13):
    """Extract MFCC mean and std across frames."""
    mfcc, _ = v_melcepst(signal, fs, 'M0', n_cep - 1)  # M0 includes c0
    return np.concatenate([mfcc.mean(axis=0), mfcc.std(axis=0)])


def extract_teo_features(signal, fs, n_bands=13):
    """Extract TEO sub-band energy features.
    
    Applies the Teager energy operator per frame, then groups
    the FFT bins into sub-bands and computes log energy per band.
    Returns mean and std across frames.
    """
    frame_len = int(0.025 * fs)
    frame_hop = int(0.010 * fs)
    frames, _, _ = v_enframe(signal, frame_len, frame_hop)
    win = v_windows(3, frame_len).flatten()
    
    teo_features = []
    for frame in frames:
        # Apply TEO to the windowed frame
        teo = v_teager(frame * win)
        # FFT of TEO signal, take magnitude
        spec = np.abs(np.fft.rfft(teo))
        # Group into sub-bands
        n_bins = len(spec)
        band_edges = np.linspace(0, n_bins, n_bands + 1, dtype=int)
        band_energies = np.array([
            np.log(np.sum(spec[band_edges[i]:band_edges[i+1]]**2) + 1e-10)
            for i in range(n_bands)
        ])
        teo_features.append(band_energies)
    
    teo_features = np.array(teo_features)
    return np.concatenate([teo_features.mean(axis=0), teo_features.std(axis=0)])


# Extract all features
mfcc_feats, teo_feats, labels = [], [], []

for _, row in df.iterrows():
    signal, fs = sf.read(row['filepath'])
    # Convert to mono if stereo
    if signal.ndim > 1:
        signal = signal.mean(axis=1)
    
    mfcc_feats.append(extract_mfcc_features(signal, fs))
    teo_feats.append(extract_teo_features(signal, fs))
    labels.append(row['emotion'])

X_mfcc = np.array(mfcc_feats)
X_teo = np.array(teo_feats)
X_both = np.hstack([X_mfcc, X_teo])
y = np.array(labels)

print(f"MFCC features:      {X_mfcc.shape}")
print(f"TEO features:       {X_teo.shape}")
print(f"MFCC + TEO features: {X_both.shape}")

## Train and evaluate

We use a Random Forest with 5-fold stratified cross-validation. Stratified folds ensure each fold has the same proportion of each emotion.

We compare three runs:
- MFCC only
- TEO only
- MFCC + TEO combined

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for name, X in [('MFCC', X_mfcc), ('TEO', X_teo), ('MFCC + TEO', X_both)]:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    y_pred = cross_val_predict(clf, X_scaled, y_enc, cv=cv)
    acc = np.mean(y_pred == y_enc)
    results[name] = {'y_pred': y_pred, 'accuracy': acc}
    print(f"{name:12s}  accuracy: {acc:.1%}")

## Confusion matrices

Each cell shows how many utterances of a given true emotion (row) were classified as each predicted emotion (column). Diagonal = correct. Off-diagonal cells reveal which emotions get confused with each other.

Common confusions in emotion recognition: boredom/neutral (both low arousal), anger/happiness (both high arousal).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
emotion_names = le.classes_

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_enc, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=emotion_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
    ax.set_title(f"{name}\n({res['accuracy']:.1%})")

plt.tight_layout()
plt.show()

## Per-emotion breakdown (MFCC + TEO)

In [ ]:
best = results['MFCC + TEO']
print(classification_report(y_enc, best['y_pred'], target_names=emotion_names))

## Comparison: accuracy by feature set

In [ ]:
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, accs, color=['#3f51b5', '#e91e63', '#4caf50'])
ax.set_ylabel('Accuracy')
ax.set_title('Emotion recognition accuracy by feature set')
ax.set_ylim(0, 1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{acc:.1%}', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()